# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/taqadussana/ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal checks:**
- Staleness (days_since_last_update): decline rate is 0.512 (fresh, n=20,655) → 0.611 (aging,
  n=9,171) → 0.467 (stale, n=169) → 0.600 (very_stale, n=5). **MIXED** — not a clean rising
  trend, and the two "stale" buckets have too few rows (169, 5) to trust on their own. This is
  the same signal behind FlyRank's real refresh flags, but on its own it's a weaker signal
  than expected.
- Visibility (impressions_90d): decline rate is 0.389 (low, n=8,006) → 0.604 (medium, n=5,279)
  → 0.618 (high, n=6,502) → 0.581 (very_high, n=10,213). **CONFIRMED** — a clear jump once a
  page has real traffic, staying elevated across higher-visibility buckets.

**My rule (plain words):** flag a page for review if it has real traffic (500+ impressions,
where decline rates are clearly elevated) AND shows the aging pattern (90-180 days since
update, the single worst-performing staleness bucket) — leaning on the CONFIRMED signal
primarily, since staleness alone was too mixed to trust.

**Reason codes this rule can output** (one per page):
- `aging_visible_page` — 90-180 days since update AND 500+ impressions (the two worst-performing bucket combination)
- `high_visibility_monitor` — 500+ impressions but doesn't meet the aging window (still worth watching, weaker signal)
- `low_signal` — neither condition met

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://raw.githubusercontent.com/taqadussana/ML/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Signal 1: staleness (feeds FlyRank's real refresh flags)
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=[-1, 90, 180, 365, 99999],
                                  labels=["fresh(<90d)", "aging(90-180d)", "stale(180-365d)", "very_stale(365d+)"])
print("Signal 1: staleness vs decline rate")
print(df.groupby("staleness_bucket")["is_declining_label"].agg(["mean", "count"]))

# Signal 2: visibility/volume (feeds quick-win logic)
df["visibility_bucket"] = pd.cut(df["impressions_90d"], bins=[-1, 100, 500, 2000, 999999999],
                                   labels=["low(<100)", "medium(100-500)", "high(500-2000)", "very_high(2000+)"])
print("\nSignal 2: visibility vs decline rate")
print(df.groupby("visibility_bucket")["is_declining_label"].agg(["mean", "count"]))

Signal 1: staleness vs decline rate
                       mean  count
staleness_bucket                  
fresh(<90d)        0.512031  20655
aging(90-180d)     0.611057   9171
stale(180-365d)    0.467456    169
very_stale(365d+)  0.600000      5

Signal 2: visibility vs decline rate
                       mean  count
visibility_bucket                 
low(<100)          0.389208   8006
medium(100-500)    0.604281   5279
high(500-2000)     0.617964   6502
very_high(2000+)   0.581416  10213


/tmp/ipykernel_843/4193282201.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("staleness_bucket")["is_declining_label"].agg(["mean", "count"]))
/tmp/ipykernel_843/4193282201.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("visibility_bucket")["is_declining_label"].agg(["mean", "count"]))


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import os

# Reason codes based on Section 1's verdicts
def assign_reason(row):
    if 90 <= row["days_since_last_update"] <= 180 and row["impressions_90d"] >= 500:
        return "aging_visible_page"
    elif row["impressions_90d"] >= 500:
        return "high_visibility_monitor"
    else:
        return "low_signal"

df["reason_code"] = df.apply(assign_reason, axis=1)
df["action"] = np.where(df["reason_code"] == "low_signal", "no_action", "review_for_refresh")

# Continuous aging score: peaks at center of the 90-180 window, fades either side
window_center = 135
df["aging_score"] = 1 - (abs(df["days_since_last_update"] - window_center) / df["days_since_last_update"].max()).clip(0, 1)

# Continuous visibility score: log-scaled, no hard clipping
df["visibility_score"] = np.log1p(df["impressions_90d"]) / np.log1p(df["impressions_90d"].max())

df["baseline_action_score"] = 0.5 * df["aging_score"] + 0.5 * df["visibility_score"]

ranked = df.sort_values("baseline_action_score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Saved", len(ranked), "ranked rows\n")

top20 = ranked[["content_id", "baseline_action_score", "reason_code", "action",
                 "days_since_last_update", "impressions_90d"]].head(20)
print(top20.to_string())

Saved 30000 ranked rows

              content_id  baseline_action_score         reason_code              action  days_since_last_update  impressions_90d
0   content_5fe46e04994d               0.958445  aging_visible_page  review_for_refresh                     104           517715
1   content_2dba2b1f9536               0.952559  aging_visible_page  review_for_refresh                     104           443434
2   content_2c2606c5d176               0.943284  aging_visible_page  review_for_refresh                     104           347399
3   content_cb112fce36be               0.938945  aging_visible_page  review_for_refresh                     104           309910
4   content_9532f197bbc8               0.938856  aging_visible_page  review_for_refresh                     104           309192
5   content_36ff89c8214e               0.937083  aging_visible_page  review_for_refresh                     104           295097
6   content_b28d1efd668f               0.935974  aging_visible_page  rev

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Top-20 review:**

All 20 top-ranked pages share the same reason code — `aging_visible_page` — meaning each one
is 90-180 days since its last update AND has 500+ impressions in the last 90 days. Scores
range from 0.958 down to 0.917, differentiated mainly by how close each page's staleness is
to the center of the aging window (135 days) and how much traffic it's still pulling.

- **Action for all 20:** `review_for_refresh` — an editor should check these first.
- **Why they're here:** each page is stale enough to likely be losing freshness signal, but
  still visible enough that a refresh has a real audience to reach.
- **What would make a given pick wrong:** if a specific page's traffic is actually flat or
  growing (not declining) despite its age, refreshing it wastes review time on something
  that isn't broken. Similarly, if a page's topic is seasonal and its content age simply
  reflects "nothing needed to change," staleness alone is misleading.

**Confidence note:** because every one of the top 20 shares the identical reason code, this
baseline currently can't distinguish "urgent" from "merely qualifies" within that group —
that's a real limitation. A page at rank 1 (score 0.958) isn't meaningfully more urgent than
rank 20 (score 0.917); the gap is mostly driven by how close each page sits to the aging
window's center, not by how bad its actual decline is.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** every page in the top 20 is a "weak pick" in the same way — the ranking
currently can't separate genuinely urgent pages from pages that just barely qualify for the
`aging_visible_page` bucket. A future version should blend in an actual decline signal
(`trend_direction` or `is_declining_label`) rather than relying on staleness + visibility
alone, since Section 1 showed staleness on its own was only MIXED, not CONFIRMED — the
"stale" and "very_stale" buckets even had too few rows (169 and 5) to trust confidently.

**Leakage check:** confirmed no product flags (`health_score`, `priority_score`, refresh
flags, etc.) or future-window data were used anywhere in the score. Every input
(`days_since_last_update`, `impressions_90d`) is observable at the decision moment, using
only the trailing 90-day window already present in the starter dataset — nothing here peeks
at outcomes that happen after the scoring point.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.